# ArthJAX Demo

Thin wrapper around the `arthjax` package.

```bash
pip install -e .
```

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from arthjax import EconomyConfig, init_state, simulate
from arthjax.simulation.step import make_step_jit

print(f"JAX {jax.__version__} | devices: {jax.devices()}")

In [ ]:
cfg = EconomyConfig(default_num_steps=600, default_seed=42)
key = jax.random.PRNGKey(cfg.default_seed)
key, init_key = jax.random.split(key)

state = init_state(cfg, init_key)
step_jit = make_step_jit(cfg)

final_state, metrics_history = simulate(state, cfg.default_num_steps, key, cfg=cfg, step_jit=step_jit)
metrics_np = jax.tree.map(np.array, metrics_history)

print(f"GDP: ${metrics_np['gdp'][-1]:.0f}")
print(f"Inflation: {metrics_np['inflation'][-1]*100:.2f}%")
print(f"Unemployment: {metrics_np['unemployment'][-1]*100:.2f}%")

In [ ]:
from arthjax.viz.plots import plot_macro_evolution, plot_linkedin_hero

plot_macro_evolution(metrics_np, "macro_evolution_v2.png")
plot_linkedin_hero(metrics_np, "linkedin_hero.png")
print("Charts saved.")

In [ ]:
# Optional: train world model (reduce epochs for quick demo)
from arthjax.world_model.train import train_world_model
from arthjax.world_model.rollout import collect_real_trajectory, compare_rollouts, rollout_learned

wm_cfg = EconomyConfig(world_model_epochs=10, world_model_num_rollouts=3)
key = jax.random.PRNGKey(100)
key, init_key = jax.random.split(key)
state = init_state(wm_cfg, init_key)
step_jit = make_step_jit(wm_cfg)

params, normalizer, losses = train_world_model(state, step_jit, wm_cfg, key)
print(f"Final loss: {losses[-1]:.4f}")